In [1]:
import ollama 

In [2]:
def ask_model(temp, text):
    return ollama.chat(
        model="llama3.2:3b",
        messages=[
            {"role": "system", "content": "You are a concise support assistant."},
            {"role": "user", "content": text}
        ],
        options={"temperature": temp}
    )["message"]["content"]

In [3]:
text = "I was charged twice this month."

In [4]:
print("TEMP = 0.1")
for i in range(3):
    print(ask_model(0.1, text))
    print("-"*40)

TEMP = 0.1
Can you tell me more about the charges? What type of account or service were they for, and approximately how much were the duplicate charges?
----------------------------------------
Can you tell me more about the charges? What type of account or service were they for, and approximately how much were the duplicate charges?
----------------------------------------
Can you tell me more about the charges? What type of account or service were you charged for, and approximately how much were the duplicate charges?
----------------------------------------


In [5]:
print("TEMP = 1.0")
for i in range(3):
    print(ask_model(1.0, text))
    print("-"*40)

TEMP = 1.0
Can you please provide more information about the charges, such as:

* What type of charge (e.g., bill, fee, etc.)
* How did you receive the duplicate charges
* Have you contacted your billing company or a customer support representative yet?
----------------------------------------
Can you tell me more about the charges? What type of account or service were you charged for, and approximately how much was each charge? This will help me better understand your issue and provide a more accurate solution.
----------------------------------------
Can you tell me more about the charges? What were they for, and have you already contacted the company or billing department regarding the double charge?
----------------------------------------


In [6]:
tickets = [
    "I was charged twice this month.",
    "The app crashes when I open settings.",
    "Please add dark mode.",
    "I cannot log in.",
    "How do I update my profile?"
]

In [7]:
def zero_shot(ticket):
    prompt = f"""
Classify into one label:
billing, bug, feature_request, other

Ticket:
{ticket}

Label:
"""
    return ollama.chat(
        model="llama3.2:3b",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.1}
    )["message"]["content"]

In [8]:
def few_shot(ticket):
    prompt = f"""
Examples:

Ticket: I want a refund.
Label: billing

Ticket: App crashes on login.
Label: bug

Ticket: Add dark mode.
Label: feature_request

Now classify:

Ticket: {ticket}
Label:
"""
    return ollama.chat(
        model="llama3.2:3b",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.1}
    )["message"]["content"]

In [9]:
def cot(ticket):
    prompt = f"""
Think briefly then classify.

Categories:
billing, bug, feature_request, other

Ticket:
{ticket}

Give reasoning in 1 sentence then label.
"""
    return ollama.chat(
        model="llama3.2:3b",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.1}
    )["message"]["content"]

In [10]:
for t in tickets:
    print("TICKET:", t)
    print("ZERO:", zero_shot(t))
    print("FEW:", few_shot(t))
    print("COT:", cot(t))
    print("="*50)

TICKET: I was charged twice this month.
ZERO: other
FEW: Based on the examples, I would classify the ticket as follows:

Ticket: I was charged twice this month.
Label: billing error or duplicate charge (or possibly "payment issue")
COT: The issue is a billing error that resulted in an incorrect charge being made to the customer.

Classification: billing
TICKET: The app crashes when I open settings.
ZERO: bug
FEW: Based on the examples, I would classify the ticket as follows:

Ticket: The app crashes when I open settings.
Label: bug
COT: The reason for the crash is likely due to an unstable or unhandled exception within the settings module of the app.

Classification: bug
TICKET: Please add dark mode.
ZERO: feature_request
FEW: Based on the examples, I would classify the ticket as follows:

Ticket: Please add dark mode.
Label: feature_request
COT: The request to add dark mode is a feature request as it involves adding a new functionality to the system.

Classification: feature_request
T

In [11]:
def structured(ticket):
    prompt = f"""
Return ONLY JSON.

Format:
{{
  "label": "billing|bug|feature_request|other",
  "confidence": 0.0,
  "reason": "short explanation"
}}

Ticket:
{ticket}
"""
    return ollama.chat(
        model="llama3.2:3b",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.1}
    )["message"]["content"]

In [12]:
output = structured("I was charged twice this month.")
print(output)

{"label": "billing|bug", "confidence": 0.0, "reason": "short explanation"}


In [13]:
import json

try:
    data = json.loads(output)

    assert data["label"] in ["billing", "bug", "feature_request", "other"]
    assert 0 <= data["confidence"] <= 1

    print("VALID JSON")
    print(data)

except Exception as e:
    print("INVALID OUTPUT")
    print(e)

INVALID OUTPUT

